In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container {width:100% !important; }</style>"))

import glob
import pandas as pd
import numpy as np
import hvplot.pandas
import plotly.express as px
import plotly.graph_objects as go
import psutil

pd.options.mode.chained_assignment = None

import datetime as dt
%autosave 30

freq = psutil.cpu_freq()
if freq:
    print(f"Current: {freq.current} MHz")

Autosaving every 30 seconds
Current: 3275.989 MHz


In [ ]:
CPU_FREQ=3.10

import os

log_dir = os.path.join('build', 'logs')
rdtsc_df_dict = []
ttt_df_dict = []
for filename in glob.glob(os.path.join(log_dir, '*.log')):
    print('processing {}'.format(filename))
    for line in open(filename):
        tokens = line.strip().split()
        if len(tokens) != 4:
            continue

        try:
            time = tokens[0]
            tag = tokens[2]
            latency = float(tokens[3])
            latency_rdtsc = latency / CPU_FREQ
            time_datetime = pd.to_datetime(time, format='%H:%M:%S.%f')
        except:
            continue

        if ' RDTSC ' in line:
            if tokens[1] != 'RDTSC':
                continue

            rdtsc_df_dict.append({'timestamp':time, 'tag':tag, 'latency':latency_rdtsc})
        elif ' TTT ' in line:
            if tokens[1] != 'TTT':
                continue

            ttt_df_dict.append({'timestamp':time, 'tag':tag, 'latency':latency})
        
rdtsc_df = pd.DataFrame.from_dict(rdtsc_df_dict)
rdtsc_df = rdtsc_df.drop_duplicates().sort_values(by='timestamp')
rdtsc_df['timestamp'] = pd.to_datetime(rdtsc_df['timestamp'], format='%H:%M:%S.%f')

ttt_df = pd.DataFrame.from_dict(ttt_df_dict)
ttt_df = ttt_df.drop_duplicates().sort_values(by='timestamp')
ttt_df['timestamp'] = pd.to_datetime(ttt_df['timestamp'], format='%H:%M:%S.%f')

print('RDTSC records:', len(rdtsc_df))
print('TTT records:', len(ttt_df))

processing build/logs/TradeEngine_1.log


In [3]:
for tag in rdtsc_df['tag'].unique():
    print(tag)
    
    fig = go.Figure()

    t_df = rdtsc_df[rdtsc_df['tag'] == tag].copy()
    t_df = t_df[t_df['latency'] > 0]

    q_hi = t_df['latency'].quantile(0.99)
    q_lo = t_df['latency'].quantile(0.01)
    t_df = t_df[(t_df['latency'] < q_hi) & (t_df['latency'] > q_lo)]

    mean = t_df['latency'].astype(float).mean()
    print('{} has {} observations mean {}'.format(tag, len(t_df), mean))

    rolling_window = max(1, int(len(t_df) / 100))

    use_micros = False
    if mean >= 1000:
        use_micros = True
        t_df['latency'] = t_df['latency'].astype(float) / 1000

    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency'], name=tag))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency'].rolling(rolling_window).mean(), name=tag + ' mean'))
#     fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency'].rolling(rolling_window).std(), name=tag + ' std'))

    fig.update_layout(title='Performance ' + tag + ' ' + ('microseconds' if use_micros else 'nanoseconds'), height=750, width=1000, hovermode='x', legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ))
    fig.show()

Trading_MarketOrderBook_addOrder
Trading_MarketOrderBook_addOrder has 1933 observations mean 1221.0214775628726


Trading_MarketOrderBook_updateBBO
Trading_MarketOrderBook_updateBBO has 4551 observations mean 321.5161502966381


Trading_PositionKeeper_updateBBO
Trading_PositionKeeper_updateBBO has 4552 observations mean 28534.008730653666


Trading_MarketDataConsumer_recvCallback
Trading_MarketDataConsumer_recvCallback has 3628 observations mean 597904.098232386


Trading_FeatureEngine_onOrderBookUpdate
Trading_FeatureEngine_onOrderBookUpdate has 4563 observations mean 86816.46942800788


...

Trading_MarketOrderBook_removeOrder
Trading_MarketOrderBook_removeOrder has 1575 observations mean 1539.2768049155145


Trading_TradeEngine_algoOnOrderUpdate
Trading_TradeEngine_algoOnOrderUpdate has 10449 observations mean 48463.17412686505


Trading_PositionKeeper_addFill
Trading_PositionKeeper_addFill has 4422 observations mean 84304.62927298989


Trading_FeatureEngine_onTradeUpdate
Trading_FeatureEngine_onTradeUpdate has 2211 observations mean 89675.58016369764


Trading_TradeEngine_algoOnTradeUpdate
Trading_TradeEngine_algoOnTradeUpdate has 2211 observations mean 47611.03033220992


In [ ]:
HOPS = [
    ['T1_OrderServer_tcp_read', 'T2_OrderServer_LFQueue_write'],
    ['T2_OrderServer_LFQueue_write', 'T3_MatchingEngine_LFQueue_read'],
    ['T3_MatchingEngine_LFQueue_read', 'T4_MathcingEngine_LFQueue_write'], ['T3_MatchingEngine_LFQueue_read', 'T4t_MathcingEngine_LFQueue_write'],
    ['T4_MathcingEngine_LFQueue_write', 'T5_MarketDataPublisher'], ['T4t_MathcingEngine_LFQueue_write', 'T5t_OrderServer_LFQueue_read'],
    ['T5_MarketDataPublisher', 'T6_MarketDataPublisher'], ['T5t_OrderServer_LFQueue_read', 'T6t_OrderServer_TCP_write'],
    ['T7_MarketDataConsumer_UDP_read', 'T8_MarketDataConsumer_LFQueue_write'], ['T7t_OrderGateway_TCP_read', 'T8t_orderGateway_LFQueue_write'],
    ['T8_MarketDataConsumer_LFQueue_write', 'T9_TradeEngine_LFQueue_read'], ['T8t_orderGateway_LFQueue_write', 'T9t_TradeEngine_LFQueue_read'],
    ['T9_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'], ['T9t_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'],
    ['T10_TradeEngine_LFQueue_write', 'T11_OrderGateway_LFQueue_read'],
    ['T11_OrderGateway_LFQueue_read', 'T12_OrderGateway_TCP_write'],
    # exchange <-> client
    ['T12_OrderGateway_TCP_write', 'T1_OrderServer_tcp_read'],
    ['T6_MarketDataPublisher', 'T7_MarketDataConsumer_UDP_read'], ['T6t_OrderServer_TCP_write', 'T7t_OrderGateway_TCP_read'],
]

In [5]:
for tags in HOPS:
    tag_p, tag_n = tags
    print('{} => {}. {} => {}.'.format(tag_p, len(ttt_df[ttt_df['tag'] == tag_p]), tag_n, len(ttt_df[ttt_df['tag'] == tag_n])))

    fig = go.Figure()

    t_df = ttt_df[(ttt_df['tag'] == tag_n) | (ttt_df['tag'] == tag_p)]
    t_df['latency_diff'] = t_df['latency'].diff()
    t_df = t_df[t_df['latency_diff'] > 0]
    t_df = t_df[t_df.tag == tag_n]

    q_hi = t_df['latency_diff'].quantile(0.99)
    q_lo = t_df['latency_diff'].quantile(0.01)
    t_df = t_df[(t_df['latency_diff'] < q_hi) & (t_df['latency_diff'] > q_lo)]

    mean = t_df['latency_diff'].astype(float).mean()
    print('{} has {} observations mean {}'.format(tag_n, len(t_df), mean))

    rolling_window = max(1, int(len(t_df) / 100))

    unit = 'nanoseconds'
    if mean >= 1000000:
        unit = 'milliseconds'
        t_df['latency_diff'] = t_df['latency_diff'].astype(float) / 1000000
    elif mean >= 1000:
        unit = 'microseconds'
        t_df['latency_diff'] = t_df['latency_diff'].astype(float) / 1000

    tag_name = tag_p + ' -> ' + tag_n
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency_diff'], name=tag_name))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['latency_diff'].rolling(rolling_window).mean(), name=tag_name + ' mean'))

    fig.update_layout(title='performance ' + tag_name + ' ' + unit, height=750, width=1000, hovermode='x', legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ))
    fig.show()


T1_OrderServer_TCP_read => 0. T2_OrderServer_LFQueue_write => 0.
T2_OrderServer_LFQueue_write has 0 observations mean nan


T2_OrderServer_LFQueue_write => 0. T3_MatchingEngine_LFQueue_read => 0.
T3_MatchingEngine_LFQueue_read has 0 observations mean nan


T3_MatchingEngine_LFQueue_read => 0. T4_MatchingEngine_LFQueue_write => 0.
T4_MatchingEngine_LFQueue_write has 0 observations mean nan


T3_MatchingEngine_LFQueue_read => 0. T4t_MatchingEngine_LFQueue_write => 0.
T4t_MatchingEngine_LFQueue_write has 0 observations mean nan


T4_MatchingEngine_LFQueue_write => 0. T5_MarketDataPublisher_LFQueue_read => 0.
T5_MarketDataPublisher_LFQueue_read has 0 observations mean nan


...

T10_TradeEngine_LFQueue_write => 7901. T11_OrderGateway_LFQueue_read => 0.
T11_OrderGateway_LFQueue_read has 0 observations mean nan


T11_OrderGateway_LFQueue_read => 0. T12_OrderGateway_TCP_write => 0.
T12_OrderGateway_TCP_write has 0 observations mean nan


T12_OrderGateway_TCP_write => 0. T1_OrderServer_TCP_read => 0.
T1_OrderServer_TCP_read has 0 observations mean nan


T6_MarketDataPublisher_UDP_write => 0. T7_MarketDataConsumer_UDP_read => 3704.
T7_MarketDataConsumer_UDP_read has 3627 observations mean 40457505.24400331


T6t_OrderServer_TCP_write => 0. T7t_OrderGateway_TCP_read => 0.
T7t_OrderGateway_TCP_read has 0 observations mean nan


In [6]:
import session_info
session_info.show()